<a href="https://colab.research.google.com/github/davikovalski/html-portfolio/blob/main/EGC5310_EstruturasDados_S07_Avaliacao1.1_Spotify.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EGC5310 — Estruturas de Dados  
## Semana 07 — Avaliação individual 1 — Spotify

**Tempo de referência:** 120 minutos  
**Grupo:** Spotify  
**Entrega:** este arquivo `.ipynb`, preenchido, pelo Moodle.

---

### Objetivo

Nesta avaliação, o mais importante **não é programar**. Os códigos necessários para preparar os dados, construir estruturas e realizar os experimentos são fornecidos.

Você deverá principalmente:

**prever → executar → observar → explicar → decidir**

Ao longo da avaliação, o conjunto de dados permanecerá praticamente o mesmo, mas as **operações e os requisitos mudarão**.

> Uma estrutura não é “boa” ou “ruim” isoladamente. Sua adequação depende do que precisamos fazer com os dados.

## Considerações importantes

- A consulta aos materiais da disciplina e à Internet está permitida.
- A discussão com colegas também está permitida.
- **A entrega e as respostas registradas neste Notebook são individuais.**
- Não é necessário memorizar código ou tabelas de complexidade.
- Sempre que houver uma **PREVISÃO**, registre sua resposta **antes de executar** a célula seguinte.
- Não altere os códigos fornecidos, exceto quando houver uma indicação explícita.
- Evite usar ferramentas de IA generativa como substitutas do seu próprio raciocínio. Elas não são necessárias para realizar a avaliação.
- Caso utilize IA como apoio, verifique criticamente o resultado, certifique-se de que consegue explicar o que registrou e informe, ao final, em quais atividades ela foi usada e com qual finalidade.
- Respostas como **“porque é mais rápido”**, **“porque é O(1)”** ou **“porque é melhor”**, sem explicar o mecanismo, geralmente serão insuficientes.
- Edite apenas o conteúdo entre os marcadores de resposta. Não apague os comentários `RESPOSTA:...:INICIO` e `RESPOSTA:...:FIM`.

### Sugestão de ritmo

| Etapa | Tempo de referência |
|---|---:|
| Preparação e dados | 10 min |
| Atividade 1 | 10 min |
| Atividade 2 | 20 min |
| Atividade 3 | 15 min |
| Atividade 4 | 15 min |
| Atividade 5 | 10 min |
| Atividade 6 | 15 min |
| Atividade 7 | 10 min |
| Atividade 8 | 15 min |

Os tempos são apenas uma referência para ajudá-lo a administrar os **120 minutos**.


## Identificação

**Nome completo:**

<!-- RESPOSTA:IDENTIFICACAO-NOME:INICIO -->

*Davi José Kovalski Fernandes*

<!-- RESPOSTA:IDENTIFICACAO-NOME:FIM -->

**Matrícula:**

<!-- RESPOSTA:IDENTIFICACAO-MATRICULA:INICIO -->

*26101561*

<!-- RESPOSTA:IDENTIFICACAO-MATRICULA:FIM -->

# 0. Preparação — conhecendo os dados

Usaremos o **Spotify Tracks Dataset — 114K Songs Across 114 Genres**, disponível no Kaggle.

Fonte:  
https://www.kaggle.com/datasets/saichaitanyareddyai/spotify-tracks-dataset-audio-features/data

Baixe o arquivo CSV indicado na página do dataset. Na versão usada na preparação desta avaliação, o arquivo chama-se:

`spotify-tracks-dataset-clean.csv`

Se estiver usando Google Colab, faça o upload do CSV para a sessão.  
Se estiver usando Jupyter local, coloque o arquivo na mesma pasta do Notebook.

A célula a seguir aceita também o nome `dataset.csv`, caso a versão baixada utilize esse nome.

In [4]:
# Opcional no Google Colab: execute esta célula para abrir o seletor de arquivos.
# Se estiver usando Jupyter local, você pode ignorá-la.

try:
    from google.colab import files
    arquivos_enviados = files.upload()
    print("Upload concluído:", list(arquivos_enviados.keys()))
except ImportError:
    print("Ambiente local detectado. Coloque o CSV na mesma pasta do Notebook.")

Saving spotify-tracks-dataset-detailed.csv to spotify-tracks-dataset-detailed.csv
Upload concluído: ['spotify-tracks-dataset-detailed.csv']


In [5]:
from pathlib import Path
import pandas as pd
import numpy as np
import random
import time
from statistics import median
from bisect import bisect_left, bisect_right

ARQUIVOS_CANDIDATOS = [
    "spotify-tracks-dataset-detailed.csv",
    "dataset.csv",
]

arquivo_csv = next(
    (Path(nome) for nome in ARQUIVOS_CANDIDATOS if Path(nome).exists()),
    None
)

if arquivo_csv is None:
    raise FileNotFoundError(
        "CSV não encontrado. Baixe o dataset e faça upload do arquivo para "
        "a sessão do Colab ou coloque-o na mesma pasta do Notebook."
    )

spotify = pd.read_csv(arquivo_csv)

COLUNAS_NECESSARIAS = {
    "track_id",
    "artists",
    "track_name",
    "popularity",
    "energy",
    "track_genre",
}

faltantes = COLUNAS_NECESSARIAS - set(spotify.columns)
if faltantes:
    raise ValueError(
        f"O arquivo carregado não possui as colunas necessárias: {sorted(faltantes)}"
    )

print("Arquivo:", arquivo_csv)
print("Dimensões:", spotify.shape)
print("Colunas necessárias encontradas.")

Arquivo: spotify-tracks-dataset-detailed.csv
Dimensões: (114000, 20)
Colunas necessárias encontradas.


In [6]:
# Vamos trabalhar apenas com as colunas relevantes para esta avaliação.
colunas = [
    "track_id",
    "artists",
    "track_name",
    "popularity",
    "energy",
    "track_genre",
]

spotify = spotify[colunas].copy()

display(spotify.head(8))

print("\nLinhas no arquivo:", len(spotify))
print("track_id distintos:", spotify["track_id"].nunique())
print("Valores distintos em artists:", spotify["artists"].nunique())
print("Gêneros distintos:", spotify["track_genre"].nunique())
print("track_id repetidos:", spotify["track_id"].duplicated().sum())

,track_id,artists,track_name,popularity,energy,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,73,0.4610,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost - Acoustic,55,0.1660,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,57,0.3590,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Can't Help Falling In Love,71,0.0596,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,82,0.4430,acoustic
5,01MVOl9KtVTNfFiBU9I7dc,Tyrone Wells,Days I Will Remember,58,0.4810,acoustic
6,6Vc5wAMmXdKIAM7WUoEb7N,A Great Big World;Christina Aguilera,Say Something,74,0.1470,acoustic
7,1EzrEOXmMH3G43AXT1y7pA,Jason Mraz,I'm Yours,80,0.4440,acoustic



Linhas no arquivo: 114000
track_id distintos: 89741
Valores distintos em artists: 31437
Gêneros distintos: 114
track_id repetidos: 24259


### Antes de avançar

Uma linha do arquivo é um registro do dataset. Dependendo da versão, uma mesma faixa pode aparecer mais de uma vez, por exemplo associada a classificações de gênero distintas.

Para as atividades em que `track_id` será tratado como identificador de uma faixa, usaremos uma **visão canônica**, mantendo uma ocorrência por `track_id`. Isso será feito explicitamente na próxima célula.

In [7]:
faixas = (
    spotify
    .drop_duplicates(subset=["track_id"], keep="first")
    .reset_index(drop=True)
)

print("Registros no arquivo original:", len(spotify))
print("Faixas na visão por track_id:", len(faixas))

display(faixas.head(5))

Registros no arquivo original: 114000
Faixas na visão por track_id: 89741


,track_id,artists,track_name,popularity,energy,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,73,0.4610,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost - Acoustic,55,0.1660,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,57,0.3590,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Can't Help Falling In Love,71,0.0596,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,82,0.4430,acoustic


### Q0.1 — Compreensão dos dados

Sem discutir ainda qual estrutura é “melhor”:

1. O que representa uma linha da visão `faixas`?
2. Qual atributo será usado como identificador das faixas nas primeiras atividades?
3. O campo `artists` identifica necessariamente uma única música?

**Sua resposta:**

<!-- RESPOSTA:Q0.1:INICIO -->

*Uma linha representa um conjunto de dados, onde cada dado é caracterizado como um atributo. O atributo usado como identificador será o track_id, por ser único e artificial. O campo artists pode representar várias músicas de um mesmo artista, então não identifica necessariamente uma única música.*

<!-- RESPOSTA:Q0.1:FIM -->

# 1. Atividade 1 — Antes de medir, escolha

## Cenário

Um serviço recebe frequentemente um `track_id` e precisa recuperar o registro completo da música correspondente.

Considere três formas de organizar **os mesmos registros**:

**A — lista de registros**
```python
musicas_lista = registros
```

**B — lista ordenada por `track_id`**
```python
musicas_ordenadas = sorted(
    registros,
    key=lambda musica: musica["track_id"]
)
```

**C — dicionário `track_id → registro`**
```python
musicas_dict = {
    musica["track_id"]: musica
    for musica in registros
}
```

## PARE — não execute benchmark ainda

Registre primeiro suas previsões.

### Q1.1 — Escolha inicial

Qual das três representações você escolheria se o sistema realizasse **muitas consultas exatas por `track_id`**?

**Sua resposta:**

<!-- RESPOSTA:Q1.1:INICIO -->

*A representação por dicionário, usando como chave o track_id, e o outros parâmetros ligados a esta chave*

<!-- RESPOSTA:Q1.1:FIM -->

### Q1.2 — Explique o mecanismo

Explique **como** a representação escolhida localiza a faixa.

Não use apenas expressões como “é mais rápida”, “é melhor” ou “é O(1)”.

**Sua resposta:**

<!-- RESPOSTA:Q1.2:INICIO -->

*Os dicionários são convertidos em tabelas hash, então a partir de uma chave específica, é feito um cálculo da posição desta chave, podendo assim acessar seus valores.*

<!-- RESPOSTA:Q1.2:FIM -->

### Q1.3 — Propriedade necessária

Qual das alternativas depende de uma **propriedade previamente preparada e preservada** para que a estratégia de busca funcione? Que propriedade é essa?

**Sua resposta:**

<!-- RESPOSTA:Q1.3:INICIO -->

*A busca ordenada. Ela depende que os dados estejam ordenados por indices, para que se possa utilizar estratégias de busca*

<!-- RESPOSTA:Q1.3:FIM -->

### Q1.4 — Depois do mecanismo, associe o crescimento

Associe às três estratégias o comportamento de busca que você espera observar, usando quando apropriado:

`O(n)` · `O(log n)` · `O(1)` médio/esperado

Justifique brevemente cada associação.

**Sua resposta:**

<!-- RESPOSTA:Q1.4:INICIO -->

*Para listas, a ordem é 0(n) pois a lista será percorrida por inteiro. Para listas ordenadas a ordem é 0(logn) pois a lista é dividida em dois a cada passo da busca. Em dicionários a ordem é 0(1) pois a estrutura é transfomada em uma tabela hash*

<!-- RESPOSTA:Q1.4:FIM -->

# 2. Atividade 2 — Preparar também custa

Agora vamos transformar as três alternativas em estruturas concretas e medir separadamente:

1. **preparação adicional da estrutura**;
2. **custo das consultas**.

Neste ponto, `registros` **já é uma lista de dicionários**. Portanto, manter essa lista não exige reorganização adicional relevante; ordenar ou construir um `dict`, por outro lado, exige trabalho de preparação.

O experimento usa uma amostra determinística para evitar que uma quantidade muito grande de buscas sequenciais transforme a avaliação em tempo de espera.

> O objetivo não é descobrir “quem ganhou a corrida”, mas relacionar o resultado medido ao mecanismo de cada estratégia.


In [8]:
# Preparação determinística dos dados experimentais.
SEMENTE = 5310
N_BENCHMARK = min(5_000, len(faixas))

amostra = faixas.sample(
    n=N_BENCHMARK,
    random_state=SEMENTE
).reset_index(drop=True)

registros = amostra.to_dict("records")

print("Quantidade de registros no benchmark:", len(registros))

Quantidade de registros no benchmark: 5000


In [9]:
def busca_sequencial(registros, track_id):
    for musica in registros:
        if musica["track_id"] == track_id:
            return musica
    return None


def busca_binaria(registros_ordenados, track_id):
    inicio = 0
    fim = len(registros_ordenados) - 1

    while inicio <= fim:
        meio = (inicio + fim) // 2
        atual = registros_ordenados[meio]["track_id"]

        if atual == track_id:
            return registros_ordenados[meio]
        elif atual < track_id:
            inicio = meio + 1
        else:
            fim = meio - 1

    return None


def preparar_lista(registros):
    # A lista já é a representação de entrada.
    # Não há reorganização adicional.
    return registros


def preparar_ordenada(registros):
    return sorted(registros, key=lambda x: x["track_id"])


def preparar_dict(registros):
    return {musica["track_id"]: musica for musica in registros}

## PARE — Q2.1

Antes de medir, ordene mentalmente as três alternativas em relação a:

- custo de preparação;
- custo de **uma** consulta;
- custo de **muitas** consultas.

Não é necessário prever números.

**Sua resposta:**

<!-- RESPOSTA:Q2.1:INICIO -->

*Para custo de preparação, o mais simples seria uma lista, depois um dicionário e por ultimo a lista ordenada. O custo de uma consulta é o menor para lista ordenada, seguido de dicionário e lista. Para muitas consultas, o dicionário é mais eficiente, depois a lista ordenada e por ultima a lista*

<!-- RESPOSTA:Q2.1:FIM -->

In [10]:
def medir_preparacao(funcao, registros, repeticoes=7):
    tempos = []
    estrutura = None

    for _ in range(repeticoes):
        inicio = time.perf_counter()
        estrutura = funcao(registros)
        tempos.append(time.perf_counter() - inicio)

    return estrutura, median(tempos)


lista, t_lista = medir_preparacao(preparar_lista, registros)
ordenada, t_ordenada = medir_preparacao(preparar_ordenada, registros)
indice_dict, t_dict = medir_preparacao(preparar_dict, registros)

resultado_preparacao = pd.DataFrame([
    {"estrutura": "lista", "preparacao_ms": t_lista * 1000},
    {"estrutura": "lista ordenada", "preparacao_ms": t_ordenada * 1000},
    {"estrutura": "dict", "preparacao_ms": t_dict * 1000},
])

display(resultado_preparacao)

,estrutura,preparacao_ms
0,lista,0.000151
1,lista ordenada,2.251307
2,dict,0.856953


In [11]:
rng = random.Random(SEMENTE)
ids_disponiveis = [r["track_id"] for r in registros]

def gerar_consultas(qtd):
    return [rng.choice(ids_disponiveis) for _ in range(qtd)]


def medir_lote_consultas(funcao, estrutura, consultas, repeticoes=1):
    tempos = []

    for _ in range(repeticoes):
        inicio = time.perf_counter()
        for track_id in consultas:
            funcao(estrutura, track_id)
        tempos.append(time.perf_counter() - inicio)

    return median(tempos)


def busca_dict(indice, track_id):
    return indice.get(track_id)


linhas = []

for qtd in [1, 100, 10_000]:
    consultas = gerar_consultas(qtd)

    # Mais repetições nos testes curtos; uma execução no lote maior.
    reps = 7 if qtd <= 100 else 1

    linhas.append({
        "consultas": qtd,
        "lista_ms": medir_lote_consultas(
            busca_sequencial, lista, consultas, reps
        ) * 1000,
        "ordenada_ms": medir_lote_consultas(
            busca_binaria, ordenada, consultas, reps
        ) * 1000,
        "dict_ms": medir_lote_consultas(
            busca_dict, indice_dict, consultas, reps
        ) * 1000,
    })

resultado_consultas = pd.DataFrame(linhas)
display(resultado_consultas)

,consultas,lista_ms,ordenada_ms,dict_ms
0,1,0.256106,0.002953,0.000319
1,100,23.327074,0.268694,0.009700
2,10000,2006.226187,38.426125,2.511104


### Q2.2 — Confrontando previsão e evidência

O resultado experimental foi compatível com sua previsão? Aponte pelo menos **uma observação concreta** da tabela que sustente sua resposta.

**Sua resposta:**

<!-- RESPOSTA:Q2.2:INICIO -->

*Não, esperava uma velocidade maior para uma busca de lista ordenada, mas o dicionário é mais rápido para qualquer quantidade de buscas. Em relação a preparação os resultados foram consistentes*

<!-- RESPOSTA:Q2.2:FIM -->

### Q2.3 — Preparação adicional

Qual representação apresentou maior custo de **preparação adicional** no seu experimento? Que trabalho ajuda a explicar esse resultado?

Considere que `registros` já estava disponível como lista antes da comparação.


**Sua resposta:**

<!-- RESPOSTA:Q2.3:INICIO -->

*A lista ordenada apresenta o maior custo, já que cada elemento adicionado precisa ser ordenado, listas e dicionários adicionam no final.*

<!-- RESPOSTA:Q2.3:FIM -->

### Q2.4 — Uma consulta × muitas consultas

Explique por que uma decisão que parece razoável para **uma consulta** pode ser diferente da decisão para **10.000 consultas**, mesmo usando os mesmos dados.

**Sua resposta:**

<!-- RESPOSTA:Q2.4:INICIO -->

*Para uma consulta, qualquer estrutura fara um bom trabalho, para muitas consultas, algumas estruturas serão ineficientes, logo a escolha é essencial.*

<!-- RESPOSTA:Q2.4:FIM -->

### Q2.5 — Tempo medido × Big-O

A alternativa que apresentou menor tempo neste computador pode ser declarada automaticamente como a alternativa com “melhor Big-O”?

Explique a diferença entre o que o benchmark mede e o que a notação assintótica descreve.

**Sua resposta:**

<!-- RESPOSTA:Q2.5:INICIO -->

*Não pode. O benchmark mede o tempo de processamento de um processo, o Big-O mede a memória usada no processamento de um processamento.*

<!-- RESPOSTA:Q2.5:FIM -->

# 3. Atividade 3 — O `dict` venceu. Então usamos `dict`?

A necessidade mudou.

Agora o sistema recebe consultas como:

> **Encontre todas as músicas cuja popularidade esteja entre 70 e 80.**

O `dict` construído anteriormente tinha esta associação:

```text
track_id → música
```

## PARE — registre sua decisão antes do próximo experimento.

### Q3.1 — Você mantém a escolha?

Para essa nova operação, você manteria automaticamente a mesma representação escolhida para consulta exata por `track_id`?

Explique por quê.

**Sua resposta:**

<!-- RESPOSTA:Q3.1:INICIO -->

*Não. Se pudesse fazer uma lista ordenada por popularidade, a busca estaria pronta.*

<!-- RESPOSTA:Q3.1:FIM -->

In [12]:
# Duas formas de responder a consultas por intervalo de popularidade.

def faixa_popularidade_varredura(registros, minimo, maximo):
    return [
        musica
        for musica in registros
        if minimo <= musica["popularity"] <= maximo
    ]


def preparar_por_popularidade(registros):
    ordenados = sorted(registros, key=lambda x: x["popularity"])
    valores = [m["popularity"] for m in ordenados]
    return ordenados, valores


def faixa_popularidade_ordenada(indice, minimo, maximo):
    ordenados, valores = indice
    esquerda = bisect_left(valores, minimo)
    direita = bisect_right(valores, maximo)
    return ordenados[esquerda:direita]


indice_popularidade = preparar_por_popularidade(registros)

# Verificação simples: as duas estratégias devem encontrar a mesma quantidade.
r1 = faixa_popularidade_varredura(registros, 70, 80)
r2 = faixa_popularidade_ordenada(indice_popularidade, 70, 80)

print("Quantidade encontrada por varredura:", len(r1))
print("Quantidade encontrada com coleção ordenada:", len(r2))
print("Mesma quantidade?", len(r1) == len(r2))

Quantidade encontrada por varredura: 149
Quantidade encontrada com coleção ordenada: 149
Mesma quantidade? True


In [14]:
# Benchmark curto: muitas consultas por intervalo.
# O tempo de preparação da coleção ordenada é medido separadamente.

rng_intervalos = random.Random(SEMENTE)
intervalos = []

for _ in range(500):
    inicio_faixa = rng_intervalos.randint(0, 90)
    fim_faixa = inicio_faixa + 10
    intervalos.append((inicio_faixa, fim_faixa))

inicio = time.perf_counter()
indice_popularidade_teste = preparar_por_popularidade(registros)
tempo_preparacao_intervalo = time.perf_counter() - inicio

inicio = time.perf_counter()
total_varredura = 0
for minimo, maximo in intervalos:
    total_varredura += len(
        faixa_popularidade_varredura(registros, minimo, maximo)
    )
tempo_varredura_intervalos = time.perf_counter() - inicio

inicio = time.perf_counter()
total_ordenada = 0
for minimo, maximo in intervalos:
    total_ordenada += len(
        faixa_popularidade_ordenada(
            indice_popularidade_teste, minimo, maximo
        )
    )
tempo_ordenada_intervalos = time.perf_counter() - inicio

assert total_varredura == total_ordenada

display(pd.DataFrame([
    {
        "etapa": "preparar ordenação por popularidade (1 vez)",
        "tempo_ms": tempo_preparacao_intervalo * 1000,
    },
    {
        "etapa": "500 consultas por varredura",
        "tempo_ms": tempo_varredura_intervalos * 1000,
    },
    {
        "etapa": "500 consultas usando a ordenação preparada",
        "tempo_ms": tempo_ordenada_intervalos * 1000,
    },
]))


,etapa,tempo_ms
0,preparar ordenação por popularidade (1 vez),4.083565
1,500 consultas por varredura,185.297875
2,500 consultas usando a ordenação preparada,1.981994


**Nota sobre o código:** você não precisa conhecer a implementação de `bisect`. Para esta atividade, observe apenas que ele aproveita uma coleção **ordenada pela mesma variável usada na consulta** para localizar os limites do intervalo.

### Q3.2 — Por que o índice anterior não resolve diretamente?

Por que um `dict` organizado como `track_id → música` não fornece diretamente uma vantagem especial para encontrar **todas** as músicas com popularidade entre 70 e 80?

**Sua resposta:**

<!-- RESPOSTA:Q3.2:INICIO -->

*Porque o dicionário terá que ser percorrido por inteiro para avaliar todos os valores*

<!-- RESPOSTA:Q3.2:FIM -->

### Q3.3 — A chave da ordenação importa

Se uma lista estiver ordenada por `track_id`, essa propriedade é suficiente para localizar eficientemente um intervalo de `popularity`?

Explique.

**Sua resposta:**

<!-- RESPOSTA:Q3.3:INICIO -->

*Não. a ordenação por track_id só é eficiente para buscas diretas ou intervalos de track_id, se precisar avaliar valores associados ao track_id, todos os items serão percorridos *

<!-- RESPOSTA:Q3.3:FIM -->

### Q3.4 — Evidência e decisão

Observe o benchmark de consultas por intervalo.

1. O que mudou quando a ordenação por `popularity` foi preparada uma vez e reutilizada em muitas consultas?
2. Por que esse resultado **não** permite concluir que devemos manter todos os dados sempre ordenados por popularidade?


**Sua resposta:**

<!-- RESPOSTA:Q3.4:INICIO -->

*O tempo de execução diminuiu. Porque nem sempre a busca será feita por popularidade, a escolha vai depender de vários fatores.*

<!-- RESPOSTA:Q3.4:FIM -->

# 4. Atividade 4 — Uma chave pode possuir vários valores

Nova necessidade:

> **Dado um valor do campo `artists`, queremos recuperar todas as músicas associadas a esse crédito de artista(s) no dataset.**

O campo `artists` é usado aqui **como aparece no arquivo**. Não estamos tentando separar ou normalizar artistas individuais.

Observe a implementação a seguir.


In [ ]:
por_artista_incorreto = {}

for musica in registros:
    por_artista_incorreto[musica["artists"]] = musica

# Escolhemos automaticamente um valor de 'artists' que apareça várias vezes.
contagens_artistas = amostra["artists"].value_counts()
artista_exemplo = contagens_artistas.index[0]

print("Valor escolhido no campo artists:", artista_exemplo)
print("Ocorrências desse valor na amostra:", int(contagens_artistas.iloc[0]))
print("\nRegistro preservado no dicionário:")
display(pd.DataFrame([por_artista_incorreto[artista_exemplo]]))

### Q4.1 — O programa executou. A representação está correta?

O código acima executa sem erro. Isso significa que ele representa corretamente a necessidade de recuperar **todas** as músicas associadas ao mesmo valor de `artists`?

Explique o que aconteceu.


**Sua resposta:**

<!-- RESPOSTA:Q4.1:INICIO -->

*Escreva sua resposta aqui.*

<!-- RESPOSTA:Q4.1:FIM -->

In [ ]:
por_artista = {}

for musica in registros:
    artista = musica["artists"]

    if artista not in por_artista:
        por_artista[artista] = []

    por_artista[artista].append(musica)

print("Quantidade preservada agora:", len(por_artista[artista_exemplo]))
display(pd.DataFrame(por_artista[artista_exemplo]).head(10))

### Q4.2 — Composição de estruturas

Explique a diferença entre:

```text
artista → música
```

e

```text
artista → lista de músicas
```

O segundo caso exigiu abandonar o `dict`? O que mudou na representação?

**Sua resposta:**

<!-- RESPOSTA:Q4.2:INICIO -->

*Escreva sua resposta aqui.*

<!-- RESPOSTA:Q4.2:FIM -->

# 5. Atividade 5 — Comparando grupos de músicas

Agora queremos trabalhar com **grupos de faixas**.

Definiremos:

- **P**: `track_id` das músicas com popularidade maior ou igual a 80;
- **E**: `track_id` das músicas com energia maior ou igual a 0,8.

A célula seguinte representa esses dois grupos e executa quatro operações entre eles. Observe o resultado e interprete o que cada operação significa no problema.


In [ ]:
populares = set(
    faixas.loc[
        faixas["popularity"] >= 80,
        "track_id"
    ]
)

energeticas = set(
    faixas.loc[
        faixas["energy"] >= 0.8,
        "track_id"
    ]
)

print("|P| =", len(populares))
print("|E| =", len(energeticas))
print("|P ∩ E| =", len(populares & energeticas))
print("|P ∪ E| =", len(populares | energeticas))
print("|P - E| =", len(populares - energeticas))
print("|P △ E| =", len(populares ^ energeticas))

### Q5.1 — Interpretação das operações

Explique, no contexto das músicas, o que representam:

1. `populares & energeticas`
2. `populares | energeticas`
3. `populares - energeticas`
4. `populares ^ energeticas`

**Sua resposta:**

<!-- RESPOSTA:Q5.1:INICIO -->

*Escreva sua resposta aqui.*

<!-- RESPOSTA:Q5.1:FIM -->

### Q5.2 — Por que `set`?

Por que `set` é uma representação natural para este problema?

Sua resposta deve ir além de “porque é rápido”.

**Sua resposta:**

<!-- RESPOSTA:Q5.2:INICIO -->

*Escreva sua resposta aqui.*

<!-- RESPOSTA:Q5.2:FIM -->

# 6. Atividade 6 — O benchmark está dizendo a verdade?

Considere o cenário:

> Queremos construir um índice por `track_id` **uma vez** e, depois, realizar muitas consultas.

Veja dois experimentos.

### Experimento A

```python
for track_id in consultas:
    indice = {
        musica["track_id"]: musica
        for musica in registros
    }
    resultado = indice.get(track_id)
```

### Experimento B

```python
for track_id in consultas:
    resultado = busca_sequencial(registros, track_id)
```

Vamos executar uma versão pequena para observar o fenômeno.

### Q6.1 — Previsão

Antes de executar: há algo no desenho do Experimento A que pode distorcer a comparação com o cenário descrito?

Explique.

**Sua resposta:**

<!-- RESPOSTA:Q6.1:INICIO -->

*Escreva sua resposta aqui.*

<!-- RESPOSTA:Q6.1:FIM -->

In [ ]:
# Experimento propositalmente pequeno para não consumir tempo da avaliação.
dados_ruins = registros[:2_000]

rng_ruim = random.Random(SEMENTE)
ids_ruins = [m["track_id"] for m in dados_ruins]
consultas_ruins = [rng_ruim.choice(ids_ruins) for _ in range(200)]

# A: reconstrói o dict em TODA consulta.
inicio = time.perf_counter()
for track_id in consultas_ruins:
    indice = {
        musica["track_id"]: musica
        for musica in dados_ruins
    }
    _ = indice.get(track_id)
tempo_dict_mal_desenhado = time.perf_counter() - inicio

# B: busca sequencial.
inicio = time.perf_counter()
for track_id in consultas_ruins:
    _ = busca_sequencial(dados_ruins, track_id)
tempo_sequencial = time.perf_counter() - inicio

# C: cenário descrito corretamente: prepara uma vez, consulta muitas.
inicio = time.perf_counter()
indice_correto = {
    musica["track_id"]: musica
    for musica in dados_ruins
}
tempo_preparacao_correta = time.perf_counter() - inicio

inicio = time.perf_counter()
for track_id in consultas_ruins:
    _ = indice_correto.get(track_id)
tempo_consultas_correto = time.perf_counter() - inicio

pd.DataFrame([
    {
        "experimento": "A — dict reconstruído em cada consulta",
        "tempo_ms": tempo_dict_mal_desenhado * 1000
    },
    {
        "experimento": "B — buscas sequenciais",
        "tempo_ms": tempo_sequencial * 1000
    },
    {
        "experimento": "C1 — preparação única do dict",
        "tempo_ms": tempo_preparacao_correta * 1000
    },
    {
        "experimento": "C2 — consultas após preparação",
        "tempo_ms": tempo_consultas_correto * 1000
    },
])

### Q6.2 — Código correto × experimento adequado

Responda separadamente:

1. O código do Experimento A está necessariamente sintaticamente errado?
2. O resultado das consultas está necessariamente incorreto?
3. Que trabalho está sendo repetido e por que isso não representa o cenário descrito?

**Sua resposta:**

<!-- RESPOSTA:Q6.2:INICIO -->

*Escreva sua resposta aqui.*

<!-- RESPOSTA:Q6.2:FIM -->

### Q6.3 — A conclusão é válida?

Avalie a conclusão:

> **“O benchmark mostrou que `dict` não oferece vantagem sobre busca sequencial.”**

Ela é sustentada pelo Experimento A? Explique.

**Sua resposta:**

<!-- RESPOSTA:Q6.3:INICIO -->

*Escreva sua resposta aqui.*

<!-- RESPOSTA:Q6.3:FIM -->

# 7. Atividade 7 — Concorda ou discorda?

Para cada afirmação:

1. marque **uma** opção alterando `[ ]` para `[x]`;
2. justifique em poucas frases.

A marcação sozinha não é suficiente.

### Q7.1

> Se uma consulta em `dict` é `O(1)`, todas as consultas executam exatamente a mesma quantidade de trabalho.

**Marque uma opção e justifique:**

<!-- RESPOSTA:Q7.1:INICIO -->

- [ ] Concordo
- [ ] Discordo

**Justificativa:**

*Escreva sua justificativa aqui.*

<!-- RESPOSTA:Q7.1:FIM -->


### Q7.2

> Se duas soluções são `O(n)`, elas devem apresentar aproximadamente o mesmo tempo de execução.

**Marque uma opção e justifique:**

<!-- RESPOSTA:Q7.2:INICIO -->

- [ ] Concordo
- [ ] Discordo

**Justificativa:**

*Escreva sua justificativa aqui.*

<!-- RESPOSTA:Q7.2:FIM -->


### Q7.3

> Uma lista ordenada por `track_id` permite usar essa ordenação para encontrar rapidamente qualquer intervalo de `popularity`.

**Marque uma opção e justifique:**

<!-- RESPOSTA:Q7.3:INICIO -->

- [ ] Concordo
- [ ] Discordo

**Justificativa:**

*Escreva sua justificativa aqui.*

<!-- RESPOSTA:Q7.3:FIM -->


### Q7.4

> Como `dict` apresentou bom desempenho para consultar `track_id`, ele é a melhor estrutura para representar todas as operações deste dataset.

**Marque uma opção e justifique:**

<!-- RESPOSTA:Q7.4:INICIO -->

- [ ] Concordo
- [ ] Discordo

**Justificativa:**

*Escreva sua justificativa aqui.*

<!-- RESPOSTA:Q7.4:FIM -->


### Q7.5

> Quando os dados possuem repetições, usar `set` é sempre uma forma adequada de limpá-los.

**Marque uma opção e justifique:**

<!-- RESPOSTA:Q7.5:INICIO -->

- [ ] Concordo
- [ ] Discordo

**Justificativa:**

*Escreva sua justificativa aqui.*

<!-- RESPOSTA:Q7.5:FIM -->


# 8. Atividade 8 — Você precisa decidir

Nesta última atividade, não existe necessariamente uma única implementação aceitável.

Use como referência **as estruturas e estratégias trabalhadas na disciplina até a Semana 06**. Não é necessário propor estruturas ainda não estudadas.

Para cada cenário, faça uma escolha defensável e explicite:

1. **estrutura/representação escolhida**;
2. **operação predominante**;
3. **propriedade ou mecanismo que favorece essa operação**;
4. **custo, limitação ou trade-off aceito**.

O mais importante é a relação entre sua decisão e o padrão de operações.


## Cenário A — Catálogo de consulta

O catálogo é carregado uma vez no início do dia. Depois disso, são realizadas **centenas de milhares de consultas exatas por `track_id`**. Quase não ocorrem alterações.

**Estrutura/representação escolhida:**

<!-- RESPOSTA:Q8.1A:INICIO -->

*Escreva sua resposta aqui.*

<!-- RESPOSTA:Q8.1A:FIM -->

**Operação predominante:**

<!-- RESPOSTA:Q8.1B:INICIO -->

*Escreva sua resposta aqui.*

<!-- RESPOSTA:Q8.1B:FIM -->

**Propriedade ou mecanismo que favorece a operação:**

<!-- RESPOSTA:Q8.1C:INICIO -->

*Escreva sua resposta aqui.*

<!-- RESPOSTA:Q8.1C:FIM -->

**Custo, limitação ou trade-off aceito:**

<!-- RESPOSTA:Q8.1D:INICIO -->

*Escreva sua resposta aqui.*

<!-- RESPOSTA:Q8.1D:FIM -->

## Cenário B — Ingestão

Novas músicas chegam continuamente e precisam ser incorporadas rapidamente. Consultas individuais são raras. Não há necessidade permanente de manter os registros ordenados por `track_id`.

**Estrutura/representação escolhida:**

<!-- RESPOSTA:Q8.2A:INICIO -->

*Escreva sua resposta aqui.*

<!-- RESPOSTA:Q8.2A:FIM -->

**Operação predominante:**

<!-- RESPOSTA:Q8.2B:INICIO -->

*Escreva sua resposta aqui.*

<!-- RESPOSTA:Q8.2B:FIM -->

**Propriedade ou mecanismo que favorece a operação:**

<!-- RESPOSTA:Q8.2C:INICIO -->

*Escreva sua resposta aqui.*

<!-- RESPOSTA:Q8.2C:FIM -->

**Custo, limitação ou trade-off aceito:**

<!-- RESPOSTA:Q8.2D:INICIO -->

*Escreva sua resposta aqui.*

<!-- RESPOSTA:Q8.2D:FIM -->

## Cenário C — Exploração por popularidade

Os dados mudam pouco, mas o sistema executa continuamente consultas por **faixas de popularidade**, como 40–50, 50–60 ou 80–100.

**Estrutura/representação escolhida:**

<!-- RESPOSTA:Q8.3A:INICIO -->

*Escreva sua resposta aqui.*

<!-- RESPOSTA:Q8.3A:FIM -->

**Operação predominante:**

<!-- RESPOSTA:Q8.3B:INICIO -->

*Escreva sua resposta aqui.*

<!-- RESPOSTA:Q8.3B:FIM -->

**Propriedade ou mecanismo que favorece a operação:**

<!-- RESPOSTA:Q8.3C:INICIO -->

*Escreva sua resposta aqui.*

<!-- RESPOSTA:Q8.3C:FIM -->

**Custo, limitação ou trade-off aceito:**

<!-- RESPOSTA:Q8.3D:INICIO -->

*Escreva sua resposta aqui.*

<!-- RESPOSTA:Q8.3D:FIM -->

### Q8.4 — Síntese final

O dataset permaneceu praticamente o mesmo ao longo da avaliação.

**O que mudou nas suas escolhas e por que elas puderam mudar?**

**Sua resposta:**

<!-- RESPOSTA:Q8.4:INICIO -->

*Escreva sua resposta aqui.*

<!-- RESPOSTA:Q8.4:FIM -->

# Registro de apoios utilizados

Este registro **não é uma questão conceitual da avaliação**. Ele existe para tornar o processo de resolução transparente.

Se você utilizou ferramenta de IA generativa, registre sucintamente:

- em qual(is) atividade(s);
- para qual finalidade;
- como verificou ou revisou a resposta.

Se não utilizou, escreva simplesmente **“Não utilizei IA generativa.”**

**Registro:**

<!-- RESPOSTA:REGISTRO-IA:INICIO -->

*Escreva sua resposta aqui.*

<!-- RESPOSTA:REGISTRO-IA:FIM -->

# Antes de entregar

Confira:

- [ ] preenchi nome e matrícula;
- [ ] registrei as previsões **antes** dos experimentos;
- [ ] executei as células necessárias;
- [ ] respondi todas as questões;
- [ ] justifiquei as escolhas pelo mecanismo e pelas operações, não apenas por rótulos de complexidade;
- [ ] preservei os marcadores `RESPOSTA:...`;
- [ ] salvei/baixei a versão final do Notebook;
- [ ] vou entregar **este `.ipynb` preenchido** no Moodle.

---

## Fim da Avaliação 1 — Spotify

> O objetivo desta atividade não foi encontrar uma estrutura vencedora, mas justificar escolhas a partir das operações, dos custos e das propriedades necessárias.